# 08 - Gradient-Boosting Model

Trains the first real (non-baseline) 24h-ahead forecaster and compares it
against the seasonal-naive baseline from `06_baseline_model.ipynb` on the
*same* chronological holdout, so any improvement is measured fairly.

Model: `sklearn.ensemble.HistGradientBoostingRegressor` - already a
project dependency (no new library needed), handles missing values and
categorical features natively, and scales to the ~2.3M rows in the
assembled table without needing to drop rows for missing features.

New features beyond the baseline table (`hour`, `day_of_week`, `month`,
holiday/lecture flags, current weather): per-station **lag** features
(exact value at a fixed time offset in the past, e.g. "1 hour ago") and
**rolling** features (a trailing time-windowed mean, e.g. "average over
the last 24h"), both computed strictly from data at or before each row's
own timestamp - see `src/muenster_bike_forecast/modeling/lag_features.py`
for why these use exact-timestamp/time-based-window lookups rather than
positional shifts (the data has real gaps).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

# Make `src/` importable regardless of whether this notebook is run from
# `notebooks/` (the normal case) or the project root.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from muenster_bike_forecast.modeling.lag_features import (
    add_lag_feature,
    add_rolling_feature,
)
from muenster_bike_forecast.modeling.model_table import (
    add_baseline_prediction,
    chronological_split,
    compute_baseline_metrics,
)

MODEL_TABLE_PATH = PROJECT_ROOT / "data" / "raw" / "model_table" / "model_table.csv"
TEST_PERIOD = pd.Timedelta(weeks=8)

RANDOM_STATE = 0

## 1. Load the assembled feature table

Reuses `data/raw/model_table/model_table.csv` as written by
`06_baseline_model.ipynb` (one row per `(station_id, datetime)` at
15-minute resolution, 23 stations, with `total_count`, the 24h-ahead
`target_total_count`, calendar features, and current weather already
joined) - not regenerated here, to keep this notebook fast and to reuse
the exact same base table the baseline was scored on.

In [2]:
full_df = pd.read_csv(MODEL_TABLE_PATH, parse_dates=["datetime"])
full_df = full_df.sort_values(["station_id", "datetime"]).reset_index(drop=True)
print(
    f"Loaded {len(full_df):,} rows x {full_df.shape[1]} columns "
    f"from {MODEL_TABLE_PATH.relative_to(PROJECT_ROOT)}"
)
full_df.head()

Loaded 2,337,596 rows x 20 columns from data\raw\model_table\model_table.csv


,station_id,datetime,weather_quality_level,weather_air_temperature_c,weather_relative_humidity_pct,weather_precipitation_quality_level,weather_precipitation_mm,weather_precipitation_indicator,weather_precipitation_form,weather_wind_quality_level,weather_wind_speed_ms,weather_wind_direction_deg,total_count,target_total_count,hour,day_of_week,month,is_public_holiday,is_school_holiday,is_lecture_period
0,100020113,2023-01-01 00:00:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,1.0,3.0,0,6,1,True,True,True
1,100020113,2023-01-01 00:15:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,15.0,3.0,0,6,1,True,True,True
2,100020113,2023-01-01 00:30:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,16.0,5.0,0,6,1,True,True,True
3,100020113,2023-01-01 00:45:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,21.0,2.0,0,6,1,True,True,True
4,100020113,2023-01-01 01:00:00,3.0,16.7,50.0,3.0,0.0,0.0,0.0,10.0,9.4,210.0,35.0,0.0,1,6,1,True,True,True


## 2. Add lag/rolling history features

- `lag_1h`, `lag_1d`, `lag_1w`: `total_count` exactly 1 hour / 1 day / 1
  week before each row, per station, via exact-timestamp lookup
  (`add_lag_feature`). `lag_1d` in particular ("same time yesterday") is
  a natural predictor of "same time tomorrow" (the target).
- `rolling_mean_2h`, `rolling_mean_24h`: trailing time-windowed mean of
  `total_count` per station (`add_rolling_feature`, `closed="left"` so
  the current row's own value is never included in its own window) -
  captures recent trend/level beyond a single lag point.

Rows near the start of a station's coverage (or across the data's real
15-minute gaps) get null feature values where no data exists at the
exact lag offset or within the rolling window - left as null rather than
fabricated, since `HistGradientBoostingRegressor` handles missing feature
values natively.

In [3]:
LAG_SPECS = {
    "lag_1h": pd.Timedelta(hours=1),
    "lag_1d": pd.Timedelta(days=1),
    "lag_1w": pd.Timedelta(weeks=1),
}
ROLLING_SPECS = {
    "rolling_mean_2h": pd.Timedelta(hours=2),
    "rolling_mean_24h": pd.Timedelta(hours=24),
}

for feature_col, lag in LAG_SPECS.items():
    full_df = add_lag_feature(full_df, lag=lag, feature_col=feature_col)

for feature_col, window in ROLLING_SPECS.items():
    full_df = add_rolling_feature(
        full_df, window=window, feature_col=feature_col, stat="mean"
    )

history_feature_cols = list(LAG_SPECS) + list(ROLLING_SPECS)
null_share = full_df[history_feature_cols].isna().mean().mul(100).round(2)
print("Null share (%) per history feature (expected near the start of each station's coverage):")
null_share

Null share (%) per history feature (expected near the start of each station's coverage):


lag_1h              0.15
lag_1d              3.07
lag_1w              4.09
rolling_mean_2h     0.04
rolling_mean_24h    0.02
dtype: float64

## 3. Chronological train/test split

Same global 8-week cutoff strategy as the baseline notebook
(`chronological_split`), applied to the feature-augmented table - a
single cutoff derived from `max(datetime)` across *all* stations, so no
station's "future" leaks relative to another's, and the split is the same
one the baseline was evaluated on for a fair comparison.

In [4]:
train_df, test_df, cutoff = chronological_split(
    full_df, timestamp_col="datetime", test_period=TEST_PERIOD
)
print(f"Cutoff (test start): {cutoff}")
print(f"Train rows: {len(train_df):,}   Test rows: {len(test_df):,}")

# Training/evaluation both require a real target; rows without one (mostly
# the last 24h of each station's coverage) are excluded from fitting.
train_labeled = train_df.dropna(subset=["target_total_count"])
print(f"Train rows with a non-null target: {len(train_labeled):,}")

Cutoff (test start): 2026-05-11 04:45:00
Train rows: 2,223,556   Test rows: 112,000
Train rows with a non-null target: 2,157,748


## 4. Train the gradient-boosting model

Feature set:

- **Numeric**: current `total_count`, current weather (`weather_*`), the
  lag/rolling history features from step 2.
- **Categorical** (`category` dtype, passed via
  `categorical_features="from_dtype"`): `station_id`, `hour`,
  `day_of_week`, `month`, `is_public_holiday`, `is_school_holiday`,
  `is_lecture_period`.

No manual imputation or row-dropping for missing feature values -
`HistGradientBoostingRegressor` natively supports `NaN` in both numeric
and categorical features.

In [5]:
CATEGORICAL_FEATURES = [
    "station_id",
    "hour",
    "day_of_week",
    "month",
    "is_public_holiday",
    "is_school_holiday",
    "is_lecture_period",
]
NUMERIC_FEATURES = [
    "total_count",
    "weather_air_temperature_c",
    "weather_relative_humidity_pct",
    "weather_precipitation_mm",
    "weather_wind_speed_ms",
    *history_feature_cols,
]
FEATURE_COLS = NUMERIC_FEATURES + CATEGORICAL_FEATURES


def _prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    X = df[FEATURE_COLS].copy()
    for col in CATEGORICAL_FEATURES:
        X[col] = X[col].astype("category")
    return X


X_train = _prepare_features(train_labeled)
y_train = train_labeled["target_total_count"]

model = HistGradientBoostingRegressor(
    categorical_features="from_dtype",
    random_state=RANDOM_STATE,
)
model.fit(X_train, y_train)
print("Model fit on", f"{len(X_train):,}", "rows.")

Model fit on 2,157,748 rows.


## 5. Evaluate on the test set, alongside the baseline

Both the seasonal-naive baseline and the gradient-boosting model are
scored with the same `compute_baseline_metrics` function (it just
compares any prediction column against `target_total_count`), on the
identical test rows, so MAE/RMSE are directly comparable.

In [6]:
test_df = add_baseline_prediction(
    test_df, current_col="total_count", prediction_col="baseline_prediction"
)
X_test = _prepare_features(test_df)
test_df["gbm_prediction"] = model.predict(X_test)

baseline_overall = compute_baseline_metrics(
    test_df, prediction_col="baseline_prediction", target_col="target_total_count"
)
gbm_overall = compute_baseline_metrics(
    test_df, prediction_col="gbm_prediction", target_col="target_total_count"
)

comparison = pd.concat(
    [
        baseline_overall.assign(model="seasonal_naive_baseline"),
        gbm_overall.assign(model="gradient_boosting"),
    ],
    ignore_index=True,
)[["model", "group", "mae", "rmse", "n_rows"]]
comparison

,model,group,mae,rmse,n_rows
0,seasonal_naive_baseline,overall,19.670172,38.626091,106043
1,gradient_boosting,overall,14.486977,27.529106,106043


In [7]:
baseline_per_station = compute_baseline_metrics(
    test_df,
    prediction_col="baseline_prediction",
    target_col="target_total_count",
    group_col="station_id",
).set_index("group")
gbm_per_station = compute_baseline_metrics(
    test_df,
    prediction_col="gbm_prediction",
    target_col="target_total_count",
    group_col="station_id",
).set_index("group")

per_station_comparison = pd.DataFrame(
    {
        "baseline_mae": baseline_per_station["mae"],
        "gbm_mae": gbm_per_station["mae"],
    }
)
per_station_comparison["mae_improvement_pct"] = (
    100
    * (per_station_comparison["baseline_mae"] - per_station_comparison["gbm_mae"])
    / per_station_comparison["baseline_mae"]
)
per_station_comparison.sort_values("mae_improvement_pct", ascending=False)

,baseline_mae,gbm_mae,mae_improvement_pct
group,,,
100034983,24.600569,14.385627,41.523197
100034980,31.520796,18.584115,41.041731
100034982,32.945550,19.695857,40.216943
300039328,28.464537,17.285845,39.272347
100031297,57.761704,35.734165,38.135197
300037926,18.900752,12.126767,35.839767
100031300,27.053044,17.753392,34.375623
100034978,11.601011,7.894727,31.947944
100034981,12.743018,8.702049,31.711242


**Caveat on the per-station table above:** two stations regress relative
to the baseline. `300037405` is a mild, plausible regression (-8.6% MAE)
within normal model variance. `300038855` is not: its GBM MAE (36.5) is
more than double the baseline's (17.3). Digging into the raw data
explains why - this station's test-window traffic collapsed to a much
lower, heavily zero-inflated regime (test-window mean ~36, median 0,
vs. an all-time mean of ~107), i.e. a real regime shift (closure,
diversion, or sensor issue - not investigated further here) partway
through its history. The seasonal-naive baseline adapts to a regime
shift instantly, since it always predicts "whatever is happening right
now"; a single global model trained on years of that station's *prior*
(higher-traffic) history adapts more slowly. This is a genuine limitation
worth flagging for future work (e.g. recency-weighted training, or a
per-station drift check) rather than something this notebook fixes.

## 6. Feature importance

`HistGradientBoostingRegressor` does not expose a built-in
`feature_importances_` attribute, so importance is estimated via
permutation importance (drop in MAE-equivalent score when a feature is
shuffled) on a random sample of the test set, for speed.

In [8]:
sample_df = test_df.dropna(subset=["target_total_count"]).sample(
    n=min(20_000, len(test_df)), random_state=RANDOM_STATE
)
X_sample = _prepare_features(sample_df)
y_sample = sample_df["target_total_count"]

importance = permutation_importance(
    model,
    X_sample,
    y_sample,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
importance_df = (
    pd.DataFrame(
        {
            "feature": FEATURE_COLS,
            "importance_mean": importance.importances_mean,
            "importance_std": importance.importances_std,
        }
    )
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)
importance_df

,feature,importance_mean,importance_std
0,total_count,18.111847,0.096063
1,day_of_week,7.145469,0.101018
2,hour,6.424452,0.124814
3,station_id,3.681023,0.107338
4,rolling_mean_2h,2.838835,0.049525
5,lag_1w,2.808359,0.013720
6,lag_1d,0.978477,0.015364
7,rolling_mean_24h,0.263018,0.026046
8,is_public_holiday,0.252794,0.027176
9,weather_air_temperature_c,0.131135,0.018479


### Reading the top 5 features

Permutation importance here is measured directly in MAE units: it's how
much worse the model's mean-absolute-error gets (in bike-count units)
when a feature's values are randomly shuffled, so it reflects how much
predictive weight the model actually places on that feature - not just a
correlation.

| Rank | Feature | Importance (MAE degradation) |
|---|---|---|
| 1 | `total_count` | 18.11 |
| 2 | `day_of_week` | 7.15 |
| 3 | `hour` | 6.42 |
| 4 | `station_id` | 3.68 |
| 5 | `rolling_mean_2h` | 2.84 |

**1. `total_count` (current count at time t).** By far the dominant
feature. This is the *level* signal: today's overall activity (weather,
a local event, a holiday mood) tends to persist into tomorrow more than
it changes. It's essentially what the seasonal-naive baseline uses on
its own - the model has learned that "how busy is it right now" carries
most of the information about "how busy will it be same time tomorrow,"
and everything else is a correction on top of it.

**2. `day_of_week`.** Since the horizon is exactly 24h, the target's
day-of-week is a deterministic function of the current one
(`(day_of_week + 1) mod 7`) - so this feature really tells the model
*what kind of day tomorrow is*: workday (commuter rush-hour pattern) vs.
weekend (flatter, later, more leisure-shaped). Cycling demand is
strongly bimodal along this axis, so it acts as a behavioral-regime
switch rather than a smooth predictor.

**3. `hour`.** Same deterministic-shift logic (`hour` at t equals `hour`
at t+24h). This encodes the diurnal traffic curve - commuter routes have
sharp 8am/5pm peaks, leisure routes peak midday/afternoon. It is the
model's proxy for "which point on the daily demand curve are we
forecasting," a major driver of raw magnitude independent of anything
else.

**4. `station_id`.** A per-location fixed effect. Stations differ by an
order of magnitude in baseline volume and by route character (arterial
commuter corridor vs. quiet recreational path). Since this is one global
model, `station_id` lets the tree splits recalibrate scale and pattern
per station rather than forcing one shape onto all 23.

**5. `rolling_mean_2h` (mean count over the trailing 2 hours).** A
short-horizon smoothed level signal, distinct from the point-in-time
`total_count` because it damps out a single noisy 15-minute reading.
Essentially tied with `lag_1w` (2.81, the count at the same weekday/hour
one week ago) for this rank - post-2026-08-21 double-counting fix, these
two features are within noise of each other (previously `lag_1w` had a
clearer edge), so treat "5th" as "these two together" rather than a
strict ordering.

**Notably, everything past rank 6 (`lag_1d`, at 0.98) - weather,
`is_public_holiday`, `is_school_holiday`, `is_lecture_period`, `lag_1h`,
`rolling_mean_24h` - contributes almost nothing** (importance well under
1). This is a genuinely interesting result: the model has essentially
learned "recent level + calendar position (day-of-week, hour, station)"
as the whole story, and the richer signals we deliberately engineered -
weather, holiday/lecture calendars, longer rolling history - add only a
thin refinement on top. That's a useful negative result for future work:
it suggests the next real gains are more likely to come from better
history/trend features (e.g. more lag horizons, station-specific
seasonality) than from further calendar or weather enrichment, and it's
a reason to sanity-check whether the weather join is capturing the right
signal (e.g. rain *during the commute window* rather than an hourly
snapshot) before investing more there.

## Summary

- Added `src/muenster_bike_forecast/modeling/lag_features.py`
  (`add_lag_feature`, `add_rolling_feature`), both leakage-safe
  (backward-looking only, exact-timestamp/time-based-window, never
  crossing station boundaries), with unit tests in
  `tests/test_lag_features.py`.
- Trained a `HistGradientBoostingRegressor` on the same feature table as
  the baseline plus lag (1h/1d/1w) and rolling-mean (2h/24h) history
  features, using native categorical/NaN handling (no manual encoding or
  row-dropping for missing features).
- Evaluated on the identical chronological 8-week test split as
  `06_baseline_model.ipynb`, overall and per station - see the
  `comparison` and `per_station_comparison` tables above for the actual
  MAE/RMSE numbers and the per-station improvement over the seasonal
  -naive baseline.
- Permutation importance (above) shows which features the model actually
  relies on - useful signal for where to invest next (e.g. more weather
  detail, more lag horizons) versus what's not pulling weight.
- One station (`300038855`) is a clear exception where the global model underperforms the baseline, due to a real traffic regime shift late in its history (see the caveat after the per-station table) - worth a per-station drift check before trusting this model's forecasts for that station specifically.